# 05 — LSTM Autoencoder Anomaly Detection

Train an **LSTM Autoencoder** for unsupervised bearing fault detection on the CWRU dataset.

Key concepts:
- Train **only on healthy (normal) data** — no fault labels needed
- Measure reconstruction error on test data: high error → anomaly
- Calibrate detection threshold at a chosen percentile of training scores
- Evaluate with AUC-ROC, F1, confusion matrix

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

np.random.seed(42)

## 1. Load CWRU Bearing Data

In [ ]:
from datasets.cwru_loader import CWRULoader

cwru_loader = CWRULoader(
    data_dir='../datasets/data/CWRU',
    fault_sizes=['normal', '0.007', '0.014', '0.021'],
    window_size=50,
    stride=25,
    normalize=True,
)

try:
    X_all, y_all, label_names = cwru_loader.load()
    X_normal, X_fault = cwru_loader.load_split()
    cwru_loaded = True
    print(f'Loaded: normal={X_normal.shape}, fault={X_fault.shape}')
except FileNotFoundError as e:
    print(f'CWRU data not found: {e}')
    print('Using synthetic data for demonstration...')
    # Normal: low-amplitude Gaussian
    X_normal = np.random.randn(600, 50).astype(np.float32) * 0.3
    # Fault: higher amplitude with impulsive components
    X_fault  = (np.random.randn(600, 50) + 0.5 * np.random.choice([-1, 1], (600, 50))
                * np.random.exponential(1.0, (600, 50))).astype(np.float32)
    X_all = np.vstack([X_normal, X_fault])
    y_all = np.array([0] * len(X_normal) + [1] * len(X_fault))
    label_names = ['normal', 'fault']
    cwru_loaded = False

print(f'Normal windows: {len(X_normal)}  |  Fault windows: {len(X_fault)}')

In [ ]:
# Split normal data into training set (for autoencoder) and test set
n_normal_train = int(len(X_normal) * 0.7)
X_ae_train = X_normal[:n_normal_train]   # training set: healthy only
X_ae_val   = X_normal[n_normal_train:]   # val set: healthy only (monitor recon error)

# Test set: mix of normal and fault
X_test  = np.vstack([X_ae_val, X_fault])
y_test  = np.array([0] * len(X_ae_val) + [1] * len(X_fault), dtype=np.int64)

print(f'AE train: {X_ae_train.shape}  (healthy only)')
print(f'AE val:   {X_ae_val.shape}    (healthy only)')
print(f'Test:     {X_test.shape}  (mixed: {(y_test==0).sum()} normal, {(y_test==1).sum()} fault)')

## 2. Model Setup and Training

In [ ]:
from models.autoencoder_anomaly import LSTMAutoencoder

config = {
    'input_size':           1,    # univariate vibration signal
    'hidden_size':          64,
    'num_layers':           2,
    'dropout':              0.1,
    'seq_len':              50,
    'lr':                   1e-3,
    'batch_size':           128,
    'epochs':               50,
    'patience':             10,
    'threshold_percentile': 95,
}

model = LSTMAutoencoder(config)
n_params = sum(p.numel() for p in model.network.parameters())
print(f'Autoencoder parameters: {n_params:,}')

In [ ]:
# Add feature dimension: (N, 50) → (N, 50, 1)
X_ae_train_3d = X_ae_train[:, :, np.newaxis]
X_ae_val_3d   = X_ae_val[:, :, np.newaxis]

history = model.fit(
    X_ae_train_3d,
    X_val=X_ae_val_3d,
    calibrate_threshold=True,
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history['train_loss'], label='Train recon loss', color='steelblue')
if history.get('val_loss'):
    ax.plot(history['val_loss'], label='Val recon loss', color='coral')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE reconstruction loss')
ax.set_title('LSTM Autoencoder Training Curves')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

## 3. Anomaly Score Distribution

In [ ]:
scores_normal = model.predict_anomaly_score(X_ae_val[:, :, np.newaxis])
scores_fault  = model.predict_anomaly_score(X_fault[:, :, np.newaxis])

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, max(scores_normal.max(), scores_fault.max()) * 1.1, 60)

ax.hist(scores_normal, bins=bins, alpha=0.6, color='steelblue', label='Normal', density=True)
ax.hist(scores_fault,  bins=bins, alpha=0.6, color='coral',     label='Fault',  density=True)

if model.threshold is not None:
    ax.axvline(model.threshold, color='red', linewidth=2, linestyle='--',
               label=f'Detection threshold ({model.threshold:.4f})')

ax.set_xlabel('Reconstruction Error (MSE)')
ax.set_ylabel('Density')
ax.set_title('Anomaly Score Distribution — Normal vs Fault')
ax.legend()
plt.tight_layout()
plt.show()

## 4. ROC Curve and Classification Metrics

In [ ]:
from evaluation.fault_metrics import (
    classification_metrics, plot_confusion_matrix, plot_roc_curve
)

X_test_3d = X_test[:, :, np.newaxis]
scores_test = model.predict_anomaly_score(X_test_3d)
labels_pred = model.classify(X_test_3d)

metrics = classification_metrics(y_test, labels_pred, scores_test, average='binary')

fig_roc = plot_roc_curve(
    y_test, scores_test,
    title='LSTM Autoencoder ROC Curve — CWRU Bearing Fault Detection'
)
plt.show()

fig_cm = plot_confusion_matrix(
    y_test, labels_pred,
    class_names=['Normal', 'Fault'],
    title='Confusion Matrix'
)
plt.show()

## 5. Threshold Sensitivity Analysis

In [ ]:
from sklearn.metrics import f1_score

percentiles = range(80, 100)
f1_scores = []

for p in percentiles:
    thresh = np.percentile(scores_normal, p)
    preds_p = (scores_test > thresh).astype(int)
    f1_scores.append(f1_score(y_test, preds_p, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(percentiles), f1_scores, 'o-', color='steelblue', linewidth=2)
best_p = list(percentiles)[np.argmax(f1_scores)]
ax.axvline(best_p, color='red', linestyle='--', label=f'Best: p{best_p} (F1={max(f1_scores):.3f})')
ax.set_xlabel('Threshold percentile (of normal training scores)')
ax.set_ylabel('F1 Score')
ax.set_title('F1 vs. Threshold Percentile')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Reconstruction Visualisation

In [ ]:
# Compare reconstruction quality for normal vs faulty
idx_normal = np.where(y_test == 0)[0][0]
idx_fault  = np.where(y_test == 1)[0][0]

recons = model.predict(X_test_3d[[idx_normal, idx_fault]])

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for i, (label, color) in enumerate(zip(['Normal', 'Fault'], ['steelblue', 'coral'])):
    orig  = X_test[i if i == 0 else idx_fault]
    recon = recons[i, :, 0]
    
    # Use correct original
    if i == 0:
        orig = X_test[idx_normal]
    else:
        orig = X_test[idx_fault]
        recon = recons[1, :, 0]

    axes[i].plot(orig,  color=color, linewidth=1.5, label='Original', alpha=0.8)
    axes[i].plot(recon, color='black', linewidth=1.2, linestyle='--', label='Reconstructed', alpha=0.7)
    
    score = np.mean((orig - recon) ** 2)
    axes[i].set_title(f'{label} signal — Reconstruction Error (MSE): {score:.5f}')
    axes[i].legend()

axes[-1].set_xlabel('Sample index')
fig.suptitle('LSTM Autoencoder: Reconstruction Quality', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

- The autoencoder learns to reconstruct **healthy** signals with low error
- **Faulty** signals have distinctly higher reconstruction errors — enabling detection
- Threshold can be tuned at deployment time without retraining
- **Key advantage:** No fault labels required — works in any new environment

**Next:** [06 — Benchmarks Comparison](06_benchmarks_comparison.ipynb)